# SupplyMind AI — Preprocessing

This notebook converts confirmed EDA decisions into the shared production
preprocessing pipeline.

In [ ]:
# -------------------
# Imports
# -------------------

from pathlib import Path

import pandas as pd

from supplymind.features.predictions.application.dataset import (
    load_tabular_dataset,
    normalize_column_names,
)
from supplymind.features.predictions.application.validation import (
    validate_binary_target,
    validate_required_columns,
    validate_timestamp_column,
)
from supplymind.features.predictions.ml.cleaning import clean_shipment_data
from supplymind.features.predictions.ml.features import add_datetime_features
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.splitting import (
    temporal_train_validation_test_split,
)

In [ ]:
# -------------------
# Configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/YOUR_DATASET_FILE.csv")
TARGET_COLUMN = "REPLACE_WITH_CONFIRMED_TARGET"
TIMESTAMP_COLUMN = "REPLACE_WITH_CONFIRMED_TIMESTAMP"

NUMERICAL_FEATURES = [
    # Add only features confirmed as available at prediction time.
]

CATEGORICAL_FEATURES = [
    # Add only features confirmed as available at prediction time.
]

In [ ]:
# -------------------
# Load and prepare
# -------------------

df = load_tabular_dataset(DATASET_PATH)
df = normalize_column_names(df)
df = clean_shipment_data(df)
df = add_datetime_features(df, [TIMESTAMP_COLUMN])

In [ ]:
# -------------------
# Validate data contract
# -------------------

required_columns = (
    [TARGET_COLUMN, TIMESTAMP_COLUMN]
    + NUMERICAL_FEATURES
    + CATEGORICAL_FEATURES
)

validation_results = [
    validate_required_columns(df, required_columns),
    validate_binary_target(df, TARGET_COLUMN),
    validate_timestamp_column(df, TIMESTAMP_COLUMN),
]

for result in validation_results:
    print(result)
    assert result.is_valid, result.errors

In [ ]:
# -------------------
# Temporal split
# -------------------

split = temporal_train_validation_test_split(
    df,
    TIMESTAMP_COLUMN,
)

print("Train:", split.train.shape)
print("Validation:", split.validation.shape)
print("Test:", split.test.shape)

In [ ]:
# -------------------
# Features and target
# -------------------

feature_columns = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

X_train = split.train[feature_columns]
y_train = split.train[TARGET_COLUMN]

X_validation = split.validation[feature_columns]
y_validation = split.validation[TARGET_COLUMN]

X_test = split.test[feature_columns]
y_test = split.test[TARGET_COLUMN]

In [ ]:
# -------------------
# Build preprocessor
# -------------------

preprocessor = build_preprocessor(
    numerical_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    scale_numerical=True,
)

transformed_train = preprocessor.fit_transform(X_train)
transformed_validation = preprocessor.transform(X_validation)

print("Transformed train shape:", transformed_train.shape)
print("Transformed validation shape:", transformed_validation.shape)

## Preprocessing checkpoint

Before continuing, confirm:

- no leakage features are present
- the test set has not been used for model selection
- preprocessing is fitted on training data only
- unknown categories are safely handled
- all candidate models will receive the same feature contract